# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields
record_sets = [rs for rs in getattr(metadata, 'record_sets', [])]
if not record_sets:
    # Fallback for 'recordSet' key if 'record_sets' is not loaded with underscores
    record_sets = getattr(metadata, 'recordSet', [])

record_set_ids = []
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")
    record_set_ids.append(rs['@id'])
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")

# Show example records for the first record set
if record_set_ids:
    print(f"\nExample records from '{record_set_ids[0]}':")
    try:
        for i, row in enumerate(dataset.records(record_set=record_set_ids[0])):
            print(row)
            if i >= 2:
                break
    except Exception as exc:
        print(f"Error loading records: {exc}")
else:
    print('No record sets found!')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If not found automatically above, manually specify record set IDs for the dataset.
# For this dataset, record_sets will be extracted from metadata or set manually as needed.
if not record_set_ids:
    # Add a known @id for demonstration (change if dataset structure is updated)
    record_set_ids = [
        # Example: '/recordsets/0',
    ]
assert record_set_ids, 'Record sets must not be empty!'
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id}")
        print(df.columns.tolist())
        display(df.head())
    except Exception as exc:
        print(f"Failed to load DataFrame for {record_set_id}: {exc}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming distributions, or grouping data to prepare for further analysis.

In [ ]:
import numpy as np

# Set the primary record set to analyze
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(record_set_id, pd.DataFrame())

if not df.empty:
    print(f"Columns in DataFrame: {df.columns.tolist()}")
    # Pick a numeric field for analysis (e.g., 'MW' for molecular weight, by @id if available)
    numeric_field = None
    for col in df.columns:
        if col.lower() in ['mw', 'molecular_weight', 'Coverage', 'Abundance', 'pI'] or df[col].dtype in [np.float64, np.int64]:
            numeric_field = col
            break
    if not numeric_field:
        print('No numeric field found; please choose a suitable numeric field displayed above for analysis.')
    else:
        print(f"Using numeric field for EDA: {numeric_field}")
        # Remove obvious outliers or filter records
        threshold = df[numeric_field].mean() + 2*df[numeric_field].std()
        filtered_df = df[df[numeric_field] < threshold]
        print(f"Records with {numeric_field} less than {threshold}:")
        display(filtered_df[[numeric_field]].head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (first 5 rows):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by a categorical field, e.g., 'Sample' or sample ID
        group_field = None
        candidate_fields = ['Sample', 'sample', 'Condition', 'condition']
        for field in candidate_fields:
            if field in df.columns:
                group_field = field
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print('No suitable group field found for grouping.')
else:
    print('No DataFrame loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field:
    # Histogram of the chosen numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping is available, boxplot by group_field
    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded, overviewed, explored, and visualized the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` Python package.
* Using Croissant's machine-readable schema, we programmatically identified the available record sets, fields, and loaded data for analysis.
* We performed basic EDA (filtering, normalization, grouping) and visualized numerical distributions.
* For a more advanced analysis, refer to the dataset documentation and consider additional domain-specific preprocessing.
